# 08 · Adapter-rank ablation

**What this notebook decides.** Whether adapter capacity is the binding
constraint. Rank is the cheapest knob available, which is exactly why it gets
turned before anyone checks whether it is the one that matters.

The dataset, recipe and seed are held fixed; only `lora_rank` varies.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    artifact,
    environment_report,
    load_records,
    preflight,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

In [ ]:
from dataclasses import asdict

from qwen_text2sql.experiments import rank_ablation_plan

pd.DataFrame([asdict(cell) for cell in rank_ablation_plan()])

### The confound to check first

PEFT scales the LoRA update by `alpha / rank`. Sweeping rank with `alpha` fixed
changes the effective step size at the same time as the capacity, so a flat curve
could mean "capacity does not matter" or "the two effects cancelled".

`scripts/run_sweep.py` handles this: it scales `alpha` with rank so that
`alpha / rank` stays at whatever the base configuration chose, unless
`use_rslora` is set, in which case alpha is left alone because rsLoRA rescales by
`1 / sqrt(rank)` itself. The table below shows what the sweep will actually run,
alongside what a fixed alpha would have given.

In [ ]:
from qwen_text2sql.config import load_config

config = load_config(ROOT / "configs/qwen35_4b_qlora.yaml")
RANKS = (4, 8, 16, 32, 64)
scale = config.lora.alpha / config.lora.rank

pd.DataFrame({"lora_rank": RANKS}).assign(
    # What the sweep runs.
    alpha_used=lambda frame: (
        config.lora.alpha if config.lora.use_rslora else (frame["lora_rank"] * scale).round()
    ),
    scaling_used=lambda frame: (
        config.lora.alpha / frame["lora_rank"]
        if config.lora.use_rslora
        else pd.Series(scale, index=frame.index)
    ),
    # What a fixed alpha would have given: a sixteen-fold swing in step size.
    scaling_if_alpha_were_fixed=lambda frame: config.lora.alpha / frame["lora_rank"],
    rslora_enabled=config.lora.use_rslora,
)

```bash
poetry run python scripts/run_sweep.py --kind rank_ablation \
  --config configs/qwen35_4b_qlora.yaml \
  --train-data data/processed/bird_train.jsonl \
  --validation-data data/processed/bird_validation.jsonl
```

In [ ]:
pd.DataFrame(preflight("rank_ablation_summary", "baseline_records"))

In [ ]:
from qwen_text2sql.evaluation.intervals import wilson_interval

summary = pd.read_csv(artifact("rank_ablation_summary").require())

if not {"lower", "upper"}.issubset(summary.columns) and {"execution_accuracy", "n"}.issubset(
    summary.columns
):
    intervals = [
        wilson_interval(round(row.execution_accuracy * row.n), int(row.n))
        for row in summary.itertuples(index=False)
    ]
    summary = summary.assign(
        lower=[interval.lower for interval in intervals],
        upper=[interval.upper for interval in intervals],
    )
summary

In [ ]:
from qwen_text2sql.reporting import figures

baseline_accuracy = None
if artifact("baseline_records").exists:
    baseline_accuracy = float(
        load_records(artifact("baseline_records").path)["execution_match"].mean()
    )

figures.rank_ablation(summary, baseline=baseline_accuracy)

In [ ]:
ordered = summary.sort_values("lora_rank")
best = ordered.loc[ordered["execution_accuracy"].idxmax()]
spread = ordered["execution_accuracy"].max() - ordered["execution_accuracy"].min()

print(f"Best rank: {int(best['lora_rank'])} at {best['execution_accuracy']:.1%}")
print(f"Spread across all ranks: {spread:.1%}")

if {"lower", "upper"}.issubset(ordered.columns):
    widest = (ordered["upper"] - ordered["lower"]).max()
    print(f"Widest single interval: {widest:.1%}")
    if spread < widest:
        print(
            "\nThe spread across ranks is smaller than the uncertainty on a single\n"
            "point. This ablation does not distinguish these ranks: prefer the\n"
            "smallest, which trains fastest and produces the smallest adapter."
        )

That last check is the one most rank sweeps skip. Picking the argmax of five
noisy point estimates and reporting it as "the optimal rank" is selection on
noise. If the spread across ranks is smaller than the interval on any single
rank, the honest conclusion is that capacity is not the binding constraint here.

## Reading this ablation

**What it shows.** How execution accuracy responds to adapter rank with
everything else held fixed.

**What it does not show.** A transferable optimum. Rank interacts with dataset
size, target modules and learning rate; this is one slice at one dataset size.
Choosing a rank on this validation set and then reporting that rank's accuracy on
the same set would violate rule 1 — use the external evaluation set for the
final number.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.